f# Info

This script performs a Random Forest classification.

The following parameters can be specified:

#### TRAIN_DATA:

The dataset the random forest will be trained on. \

N22 = Northern Bavaria 2022\
N23 = Northern Bavaria 2023\
S20 = Southern Bavaria 2020\
S22 = Southern Bavaria 2022\
ALL = all of the above\

#### VAL_DATA:

The dataset the random forest will be validated on.\

N22 = Northern Bavaria 2022\
N23 = Northern Bavaria 2023\
S20 = Southern Bavaria 2020\
S22 = Southern Bavaria 2022\
ALL = all of the above\

_If VAL_DATA == TRAIN_DATA, as 5-fold Cross Validation will be used._

#### method:

This is the sampling method:\

weighted: Solely minority class ("deadwood") gets oversampled. Specify **weight** as well. Specify **total_n** (number of observations for the model) as well.\
undersampling: All the classes get undersampled to the number of the minority class.\
oversampling: All the classes get oversampled to the number of the majority class ("undisturbed"). Specify, XXXXx as well.




# 1 Import packages and functions

In [2]:
import xarray as xr
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold

In [3]:
from plot_functions import *
from read_functions import *
from datahandling_functions import *
from analysis_functions import *

# 2 Read

In [4]:
chosen_bands = ["blue", "green", "red", "rededge1", "rededge3", "nir", "swir1", "swir2", "ndvi", "savi", "ndwi", "bsi", "contrast"] #options: "blue", "green", "red", "rededge1", "rededge2", "rededge3", "nir", "nir_narrow", "swir1", "swir2", "ndvi", "savi", "ndwi", "bsi", "nssi", "contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"
TRAIN_DATA = "ALL"
VAL_DATA = "ALL"
method = "weighted"
# if method = weight:
weight = 4
FIT = "d"

## 2.1 Read Train

In [5]:
if TRAIN_DATA == "ALL":

    df1 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n22.tif")
    df2 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n23.tif")
    df3 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s20.tif")
    df4 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s22.tif")
    df = pd.concat([df1, df2, df3, df4], axis=0, ignore_index=True)
    df = df.reset_index()
    final_selection = ["x", "y"] + [b for b in chosen_bands if b not in ["x", "y", "trainclass"]] + ["trainclass", "region", "region_class"]

if TRAIN_DATA != "ALL":

    df = open_to_pd_df(f"C:/Users/miles/OneDrive/Dokumente/ROOT/trainingdata_collection/trainingdata_withindices/{TRAIN_DATA.lower()}.tif")

    final_selection = ["x", "y"] + [b for b in chosen_bands if b not in ["x", "y", "trainclass"]] + ["trainclass"]

df.columns.name = None
df = df.loc[:, ~df.columns.duplicated()]
train_df = df[final_selection]

In [6]:
statistics(df)


--- Dataset Statistics ---
Number of total valid pixels per region and class:
trainclass      1     2       3
region                         
n22         25700  1510  342093
n23         34140   969   97405
s20          7230   353  103185
s22          3234    22  464211

Total valid pixels per region/class (in % of total dataset):
trainclass       1       2        3
region                             
n22         2.38 %  0.14 %  31.67 %
n23         3.16 %  0.09 %   9.02 %
s20         0.67 %  0.03 %   9.55 %
s22          0.3 %   0.0 %  42.98 %


## 2.2 Read Val


In [7]:
if VAL_DATA != TRAIN_DATA:

    val_df = open_to_pd_df(f"C:/Users/miles/OneDrive/Dokumente/ROOT/trainingdata_collection/trainingdata_withindices/{VAL_DATA.lower()}.tif")
    val_df = val_df.reset_index()
    val_df.columns.name = None
    val_df = val_df.loc[:, ~val_df.columns.duplicated()]
    final_selection = ["x", "y"] + [b for b in chosen_bands if b not in ["x", "y", "trainclass"]] + ["trainclass"]
    val_df = val_df[final_selection]

    forestclass_test = val_df["trainclass"]
    pred_test = val_df.drop(columns=["trainclass", "region", "regionclass"])

    forestclass_train = train_df["trainclass"]
    pred_train = train_df.drop(columns=["trainclass", "region", "regionclass"])


if VAL_DATA == TRAIN_DATA:

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pred = df.drop(columns=['trainclass', 'region', 'region_class', "index"])
    forestclass = df['trainclass']
    strat_col = df['region_class']

In [8]:
print("klassen im training")
strat_df = strat_col.value_counts()
print(strat_df / strat_df.sum().sum()*100)

klassen im training
region_class
s22_3    42.980431
n22_3    31.673753
s20_3     9.553707
n23_3     9.018547
n23_1     3.160959
n22_1     2.379515
s20_1     0.669412
s22_1     0.299430
n22_2     0.139808
n23_2     0.089718
s20_2     0.032684
s22_2     0.002037
Name: count, dtype: float64


# 3. TRAIN

In [9]:
if VAL_DATA == TRAIN_DATA:
    all_accuracies = []
    all_f1s = []
    all_cms = []

    # Pre-allocate for performance and alignment
    cv_predictions = np.zeros(len(forestclass))
    all_coordinates = np.zeros((len(pred), 2))

    print(f"{'Fold':<10} | {'Accuracy':<10} | {'F1 (Weighted)':<15}")
    print("-" * 45)

    for i, (train_idx, test_idx) in enumerate(skf.split(pred, strat_col)):

        # SPLIT
        pred_train, pred_test = pred.iloc[train_idx], pred.iloc[test_idx]
        forestclass_train, forestclass_test = forestclass.iloc[train_idx], forestclass.iloc[test_idx]

        all_coordinates[test_idx] = pred_test[["x", "y"]].values
        pred_test_model = pred_test.drop(columns=["x", "y"])

        # SAMPLING
        train = pred_train.assign(trainclass=forestclass_train).drop(columns=["x", "y"])
        train = rf_sample(train, method=method, weight=weight)
        forestclass_train_sampled = train["trainclass"]
        pred_train_sampled = train.drop("trainclass", axis=1)


        # TRAINING AND PREDICTING
        rf = randomForestClass(ntrees=10, pred_train=pred_train_sampled, forestclass_train=forestclass_train_sampled, FIT=FIT)
        predictions = rf.predict(pred_test_model)

        cv_predictions[test_idx] = predictions

        acc = accuracy_score(forestclass_test, predictions)
        all_accuracies.append(acc)
        print(f"{i:<10} | {acc:<10.4f}")

else:
    all_coordinates = pred_test[["x", "y"]].values

    rf = randomForestClass(ntrees=750, pred_train=pred_train.drop(columns=["x", "y"]), forestclass_train=forestclass_train)
    cv_predictions = rf.predict(pred_test.drop(columns=["x", "y"]))



Fold       | Accuracy   | F1 (Weighted)  
---------------------------------------------
unbalanced scikit learn mode was used.
0          | 0.9719    
unbalanced scikit learn mode was used.
1          | 0.9720    
unbalanced scikit learn mode was used.
2          | 0.9728    
unbalanced scikit learn mode was used.
3          | 0.9721    
unbalanced scikit learn mode was used.
4          | 0.9721    


# Mapping the prediction

In [10]:
if VAL_DATA == TRAIN_DATA and VAL_DATA == "ALL":

    templates = {
        "n22": r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n22.tif",
        "n23": r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n23.tif",
        "s20": r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s20.tif",
        "s22": r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s22.tif"}

    output_folder = r"C:\Users\miles\OneDrive\Dokumente\ROOT\models\outputs"

    map_predictions(
        all_coordinates = all_coordinates,
        cv_predictions = cv_predictions,
        region_series = df['region'],
        templates = templates,
        output_dir = output_folder)

Successfully saved aligned raster for n22 at C:\Users\miles\OneDrive\Dokumente\ROOT\models\outputs\prediction_n22.tif
Successfully saved aligned raster for n23 at C:\Users\miles\OneDrive\Dokumente\ROOT\models\outputs\prediction_n23.tif
Successfully saved aligned raster for s20 at C:\Users\miles\OneDrive\Dokumente\ROOT\models\outputs\prediction_s20.tif
Successfully saved aligned raster for s22 at C:\Users\miles\OneDrive\Dokumente\ROOT\models\outputs\prediction_s22.tif


In [11]:
print(pred["x"])
print(pred["y"])

0          4.421881e+06
1          4.421891e+06
2          4.421861e+06
3          4.421871e+06
4          4.421881e+06
               ...     
1080047    4.595834e+06
1080048    4.595844e+06
1080049    4.595854e+06
1080050    4.595864e+06
1080051    4.595874e+06
Name: x, Length: 1080052, dtype: float64
0          3.029447e+06
1          3.029447e+06
2          3.029437e+06
3          3.029437e+06
4          3.029437e+06
               ...     
1080047    2.836187e+06
1080048    2.836187e+06
1080049    2.836187e+06
1080050    2.836187e+06
1080051    2.836187e+06
Name: y, Length: 1080052, dtype: float64


# 4. PLOT

In [12]:
if VAL_DATA == TRAIN_DATA:

    importances = pd.Series(rf.feature_importances_, index=pred.columns)

    fig = plot_cross_model_results(
        y_true=forestclass,
        y_pred=cv_predictions,
        feature_importances=importances,
        train_mode=TRAIN_DATA,
        val_mode=VAL_DATA,
        class_names=['Disturbed', 'Deadwood', 'Undisturbed']
    )

    plt.show()

else:

    fig = plot_model_results(
        y_true=forestclass_test,
        y_pred=predictions,
        rf_model=rf,
        feature_names=chosen_bands,
        train_mode=TRAIN_DATA,
        val_mode=VAL_DATA,)
    plt.show()

ValueError: Length of values (21) does not match length of index (23)